# Phase 6 §7 — the same GCG, against a top-quartile-truncated metric

Phase 6 §5–§6 maximised `score = Σ_v p_t(v)·cos(e_v, e_bridge)` over the **whole**
151,936-token vocabulary and moved only the space it was optimised on. One suspicion that
leaves open: the optimiser may be buying its score in the tail — pushing probability onto
bridge-ish tokens that are never actually emitted, where 114k tokens' worth of gradient
lives but ~no probability mass does.

This run truncates the sum at each position to the **top 25% of the vocabulary by
probability** (K = 37,984), renormalised:

```
score_t = SUM_{v in top-K} p_t(v)·cos(e_v, e_bridge)  /  SUM_{v in top-K} p_t(v)
```

The *value* should barely move — sub-1-bit entropy means the discarded tail carries almost
no mass. The *gradient* changes a lot: three quarters of the vocabulary can no longer be
pushed on at all. Same backbone, same blocklist, same tuned hyperparameters as phase 6's
winning trial (k=53, n_mut=7, n_top=512, n_cand=256, pool=full, suffix, n_new=48,
refresh_every=2, init=repeat, seed=1), so the only changed variable is the objective.

In [2]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 6 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 145.9 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [3]:
# Environment — must run BEFORE anything imports huggingface_hub.
#
# 1. HF_HUB_DISABLE_XET: the Xet backend hung this notebook dead — small JSON files
#    completed, then "Downloading bytes: 0.00B / Fetching 5 files: 0/5" sat at zero
#    for 5+ minutes on the safetensors shards. Same failure phase 3 hit; see
#    phase3/README.md's operational note. Falls back to plain HTTPS range requests.
# 2. HF_TOKEN: read from the Colab secrets vault. Needs this notebook's per-secret
#    "Notebook access" toggle ON (key icon, left sidebar), or the fetch blocks on a
#    grant prompt and times out with "Secrets can only be fetched when running from
#    the Colab UI". Unauthenticated works for public repos but is rate-limited.
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN present:", bool(tok))
except Exception as e:
    print(f"HF_TOKEN unavailable ({type(e).__name__}) — continuing unauthenticated")

print("HF_HUB_DISABLE_XET:", os.environ["HF_HUB_DISABLE_XET"])

HF_TOKEN present: True
HF_HUB_DISABLE_XET: 1


In [4]:
# Load Qwen3-8B (bf16 where supported, else fp16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"

# T4 is Turing (SM 7.5): no bf16. A100/L4 are Ampere+ and worth taking.
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="cuda").eval()   # `torch_dtype` is deprecated in v5

print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("layers:", model.config.num_hidden_layers, "| d_model:", model.config.hidden_size,
      "| vocab:", model.config.vocab_size)
print(f"weights: {sum(p.numel() for p in model.parameters())/1e9:.2f} B | "
      f"GPU total: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB | "
      f"allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded: Qwen/Qwen3-8B
device: cuda:0 | dtype: torch.bfloat16
layers: 36 | d_model: 4096 | vocab: 151936
weights: 8.19 B | GPU total: 39.5 GiB | allocated: 15.3 GiB


In [5]:
# === Cosine to ' bridge' in all four spaces (phase 6 §1's construction) ===
import torch, torch.nn.functional as F

TARGET = " bridge"
tgt = tokenizer(TARGET, add_special_tokens=False).input_ids
assert len(tgt) == 1, f"{TARGET!r} is not a single token: {tgt}"
TGT_ID = tgt[0]

assert model.config.tie_word_embeddings is False, "embeddings are tied; in/out are identical"
SPACES = {"in": model.model.embed_tokens.weight, "out": model.lm_head.weight}

COS = {}
for name, W in SPACES.items():
    E = W.detach().float()
    for kind in ("raw", "cent"):
        X = E - E.mean(0, keepdim=True) if kind == "cent" else E
        Xn = F.normalize(X, dim=-1)
        COS[f"{name}.{kind}"] = (Xn @ Xn[TGT_ID]).contiguous()
        del Xn, X
    del E
    torch.cuda.empty_cache()
KEYS = list(COS)
V = model.config.vocab_size

K_KEEP = V // 4                       # top 25% of the vocabulary by probability
print(f"target {TARGET!r} -> id {TGT_ID} | vocab {V} | K_KEEP {K_KEEP}")
print("uniform baseline:", {k: round(COS[k].mean().item(), 4) for k in KEYS})


target ' bridge' -> id 14164 | vocab 151936 | K_KEEP 37984
uniform baseline: {'in.raw': 0.0166, 'in.cent': -0.0023, 'out.raw': 0.0184, 'out.cent': -0.0002}


In [6]:
# === The truncated metric, and whether it still separates (phase 6 §2, redone) ===
#
# Before optimising against it: does keeping only the top quartile still rank real bridge
# questions above unrelated ones? If truncation broke the separation there would be no
# point searching against it. Both metrics are reported side by side, in all four spaces.
import torch

QUERIES = [
    "what shall i do today",
    "recommend me a book",
    "how do I make friends in a new city?",
    "what should I get my brother for his birthday?",
    "tell me about bridges",
    "explain how suspension bridges work",
]

@torch.no_grad()
def whole_answer_probs(q, n_new=160):
    """greedy rollout, then one teacher-forced pass -> [T, V] answer-position probs"""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)[0]
    npr = enc.input_ids.shape[1]
    ans = out[npr:]
    lg = model(out.unsqueeze(0), **{_LTK_: len(ans) + 1}).logits[0, :-1].float()
    return lg.softmax(-1), ans

import inspect
_LTK_ = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
         else "num_logits_to_keep")

summary = {}
print(f"{'query':<46} {'space':<9} {'full':>9} {'top25%':>9} {'mass kept':>10}")
print("-" * 88)
for q in QUERIES:
    P, ans = whole_answer_probs(q)
    vals, idx = P.topk(K_KEEP, dim=-1)
    den = vals.sum(-1)
    row = {}
    for k in KEYS:
        full  = (P @ COS[k]).mean().item()
        trunc = ((vals * COS[k][idx]).sum(-1) / den).mean().item()
        row[k] = dict(full=full, trunc=trunc)
        print(f"{q if k == KEYS[0] else '':<46} {k:<9} {full:>9.4f} {trunc:>9.4f} "
              f"{den.mean().item():>10.4f}")
    row["distinct"] = len(set(ans.tolist())) / len(ans)
    row["H_bits"]   = (-(P * P.clamp_min(1e-12).log2()).sum(-1)).mean().item()
    summary[q] = row
    del P, vals, idx
    torch.cuda.empty_cache()
    print("-" * 88)

ctrl = QUERIES[:4]; brid = QUERIES[4:]
print("\nseparation check (control band vs bridge queries):")
for k in KEYS:
    for m in ("full", "trunc"):
        cb = [summary[q][k][m] for q in ctrl]; bb = [summary[q][k][m] for q in brid]
        ok = "OK  " if min(bb) > max(cb) else "FAIL"
        print(f"  {k:<9} {m:<6} controls [{min(cb):+.4f}, {max(cb):+.4f}]  "
              f"bridge [{min(bb):+.4f}, {max(bb):+.4f}]  {ok}")


query                                          space          full    top25%  mass kept
----------------------------------------------------------------------------------------
what shall i do today                          in.raw       0.0368    0.0368     1.0000
                                               in.cent      0.0198    0.0198     1.0000
                                               out.raw     -0.0216   -0.0216     1.0000
                                               out.cent     0.0546    0.0546     1.0000
----------------------------------------------------------------------------------------
recommend me a book                            in.raw       0.0359    0.0359     1.0000
                                               in.cent      0.0189    0.0189     1.0000
                                               out.raw     -0.0181   -0.0181     1.0000
                                               out.cent     0.0516    0.0516     1.0000
------------------------------

In [7]:
# === §7 — GCG machinery, metric truncated to the top quartile of the vocabulary ===
#
# Objective IS the metric: mean over answer positions of SUM_v p_t(v)*cos(e_v, e_bridge),
# in the out.cent space, TRUNCATED: at each position only the K_KEEP highest-probability
# tokens enter the sum, renormalised. Not a proxy for the metric — the thing itself. The answer is a greedy
# rollout from the current trigger, refreshed every `refresh_every` steps; the gradient
# and the accept test both score against that rollout, and a separate `true_metric`
# regenerates from scratch so progress is never read off the teacher-forced number alone.
#
# Blocklist ON (bridge in ~40 languages + top-300 embedding neighbours). Without it a
# 254-slot search just writes an English prompt injection, which measures nothing.
# Pictographs ALLOWED.
import torch, torch.nn.functional as F, random, time, inspect, unicodedata

dev = model.device
V   = model.config.vocab_size
METRIC_SPACE = "out.cent"                       # the space phase 6's tuned study used
TARGET_COS = COS[METRIC_SPACE]                  # the metric's cosine vector
_LTK = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
        else "num_logits_to_keep")

# ---------- vocab strings, decoded once ----------------------------------------
TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
NFKD   = [unicodedata.normalize("NFKD", s).casefold() for s in TOKSTR]

# ---------- structural guard ---------------------------------------------------
usable = torch.ones(V, dtype=torch.bool)
for i in set(tokenizer.all_special_ids) | set(tokenizer.get_added_vocab().values()):
    if i < V: usable[i] = False
for i, s in enumerate(TOKSTR):
    if not s.strip() or any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s):
        usable[i] = False
print(f"vocab {V} -> usable {int(usable.sum())}")

# ---------- target blocklist ---------------------------------------------------
TRANSLATIONS = [
    "bridge", "bridges", "puente", "ponte", "pont", "brucke", "brücke", "brug", "bro",
    "brú", "мост", "міст", "most", "γέφυρα", "gefyra", "köprü", "kopru", "جسر", "גשר",
    "پل", "पुल", "সেতু", "桥", "橋", "大桥", "ブリッジ", "はし", "다리", "브리지",
    "cầu", "cau", "สะพาน", "jembatan", "jambatan", "silta", "sild", "híd", "hid",
    "pod", "tilts", "tiltas", "droichead", "pons", "ponto", "daraja", "tulay",
    "ხიდი", "կամուրջ", "viaduct", "viaduc", "aqueduct", "overpass", "causeway",
    "trestle", "footbridge",
]
blocked = torch.zeros(V, dtype=torch.bool)
for i, s in enumerate(NFKD):
    if s and any(t in s for t in TRANSLATIONS):
        blocked[i] = True
n_sub = int(blocked.sum())

E_c = (model.model.embed_tokens.weight.float()
       - model.model.embed_tokens.weight.float().mean(0, keepdim=True))
E_cn = F.normalize(E_c, dim=-1)
nbrs = (E_cn @ E_cn[TGT_ID]).topk(300).indices
blocked[nbrs.cpu()] = True
del E_cn
torch.cuda.empty_cache()
print(f"blocked: {n_sub} by substring (~{len(TRANSLATIONS)} forms) + neighbours "
      f"-> {int(blocked.sum())} total ({100*int(blocked.sum())/V:.2f}% of vocab)")

WEAKNESS = E_c.norm(dim=-1).cpu()               # small norm = undertrained
del E_c
torch.cuda.empty_cache()

def build_pool(kind):
    ok = usable & ~blocked
    if kind == "full":
        return ok.clone()
    n = int(kind.replace("weak", ""))
    idx = torch.nonzero(ok).squeeze(-1)
    keep = idx[WEAKNESS[idx].argsort()[:n]]
    m = torch.zeros(V, dtype=torch.bool); m[keep] = True
    return m

# ---------- scaffold -----------------------------------------------------------
SENT = "␞"
def make_scaffold(q, position):
    content = f"{SENT} {q}" if position == "prefix" else f"{q} {SENT}"
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": content}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)
    pre, suf = text.split(SENT)
    return (tokenizer(pre, add_special_tokens=False).input_ids,
            tokenizer(suf, add_special_tokens=False).input_ids)

# ---------- objective ----------------------------------------------------------
@torch.no_grad()
def rollout(trig, PRE, SUF, n_new):
    ids = torch.tensor([PRE + trig.tolist() + SUF], device=dev)
    out = model.generate(ids, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)[0]
    return out[ids.shape[1]:]

@torch.no_grad()
def score_batch(trigs, PRE, SUF, ans, chunk=8):
    """mean_t SUM_v p_t(v) cos_v, teacher-forced on `ans`. trigs: [B, k] -> [B]"""
    B  = trigs.shape[0]
    na = len(ans)
    pre = torch.tensor(PRE, device=dev); suf = torch.tensor(SUF, device=dev)
    out = []
    for i in range(0, B, chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1),
                         ans.expand(b, -1)], dim=1)
        lg = model(seq, **{_LTK: na + 1}).logits[:, :-1].float()
        p  = lg.softmax(-1); del lg
        vals, idx = p.topk(K_KEEP, dim=-1, sorted=False); del p
        out.append(((vals * TARGET_COS[idx]).sum(-1) / vals.sum(-1)).mean(-1))
        del vals, idx
    return torch.cat(out)

def grad_onehot(trig, PRE, SUF, ans):
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), num_classes=V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE, device=dev)], oh @ E,
                     E[torch.tensor(SUF, device=dev)], E[ans]]).unsqueeze(0)
    lg = model(inputs_embeds=inp, **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    p  = lg.softmax(-1)
    vals, idx = p.topk(K_KEEP, dim=-1, sorted=False)
    ((vals * TARGET_COS[idx]).sum(-1) / vals.sum(-1)).mean().backward()
    g = oh.grad.detach().clone()
    del oh, inp, lg
    torch.cuda.empty_cache()
    return g

@torch.no_grad()
def true_metric(trig, PRE, SUF, n_new=45):
    ans = rollout(trig, PRE, SUF, n_new)
    return score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item(), ans

# ---------- the search ---------------------------------------------------------
def gcg(q="what shall i do today", k=32, position="suffix", pool_kind="weak4096",
        n_top=256, n_cand=128, n_new=45, refresh_every=4, chunk=8,
        steps=10**9, budget_s=90, seed=1, log=None):
    rng = random.Random(seed); torch.manual_seed(seed)
    PRE, SUF = make_scaffold(q, position)
    pool = build_pool(pool_kind)
    pidx = torch.nonzero(pool).squeeze(-1)
    trig = pidx[torch.randint(len(pidx), (k,), generator=torch.Generator().manual_seed(seed))]

    ans = rollout(trig, PRE, SUF, n_new)
    best_t, best_true = trig.clone(), score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item()
    t0, s = time.time(), 0
    while s < steps and time.time() - t0 < budget_s:
        g = grad_onehot(trig, PRE, SUF, ans)
        g[:, ~pool.to(dev)] = -float("inf")
        top = g.topk(min(n_top, int(pool.sum())), dim=-1).indices
        del g
        slots = torch.randint(k, (n_cand,))
        picks = top[slots, torch.randint(top.shape[1], (n_cand,))]
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        cands[torch.arange(n_cand), slots] = picks
        sc = score_batch(cands, PRE, SUF, ans, chunk=chunk)
        j = int(sc.argmax())
        trig = cands[j].cpu()
        s += 1
        if s % refresh_every == 0:
            ans = rollout(trig, PRE, SUF, n_new)
            tm, _ = score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item(), None
            if tm > best_true: best_true, best_t = tm, trig.clone()
            if log: log(s, tm, time.time() - t0)
        del cands, sc, top
        torch.cuda.empty_cache()
    tm, a = true_metric(best_t, PRE, SUF, n_new)
    return dict(trigger=best_t, true=tm, answer=tokenizer.decode(a, skip_special_tokens=True),
                steps=s, secs=time.time() - t0)

print(f"machinery ready | space={METRIC_SPACE} | K_KEEP={K_KEEP}")

vocab 151936 -> usable 148023
blocked: 375 by substring (~55 forms) + neighbours -> 659 total (0.43% of vocab)
machinery ready | space=out.cent | K_KEEP=37984


In [8]:
# === §7 — multi-slot GCG at phase 6's winning hyperparameters ===
#
# gcg2 verbatim from phase 6 §2. No Optuna re-search: the point is to change the objective
# and hold everything else fixed, so the tuned params from trial 12 are reused as-is.
import torch, time, random

def gcg2(q="what shall i do today", k=64, position="suffix", pool_kind="weak4096",
         n_top=256, n_cand=128, n_mut=1, n_new=32, refresh_every=2, chunk=16,
         budget_s=150, seed=1, init="random", log=None):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    PRE, SUF = make_scaffold(q, position)
    pool = build_pool(pool_kind); pool_d = pool.to(dev)
    pidx = torch.nonzero(pool).squeeze(-1)
    if init == "repeat":
        trig = pidx[torch.randint(len(pidx), (1,), generator=gen)].repeat(k)
    else:
        trig = pidx[torch.randint(len(pidx), (k,), generator=gen)]

    ans = rollout(trig, PRE, SUF, n_new)
    best_t = trig.clone()
    best   = score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item()
    t0, s = time.time(), 0
    while time.time() - t0 < budget_s:
        g = grad_onehot(trig, PRE, SUF, ans)
        g[:, ~pool_d] = -float("inf")
        top = g.topk(min(n_top, int(pool.sum())), dim=-1).indices
        del g
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        for _ in range(n_mut):
            slots = torch.randint(k, (n_cand,))
            picks = top[slots, torch.randint(top.shape[1], (n_cand,))]
            cands[torch.arange(n_cand), slots] = picks
        sc = score_batch(cands, PRE, SUF, ans, chunk=chunk)
        trig = cands[int(sc.argmax())].cpu()
        s += 1
        if s % refresh_every == 0:
            ans = rollout(trig, PRE, SUF, n_new)
            tm = score_batch(trig.unsqueeze(0), PRE, SUF, ans)[0].item()
            if tm > best: best, best_t = tm, trig.clone()
            if log: log(s, tm, best, time.time() - t0)
        del cands, sc, top
        torch.cuda.empty_cache()
    tm, a = true_metric(best_t, PRE, SUF, 45)
    return dict(trigger=best_t, best_tf=best, true=tm, steps=s,
                answer=tokenizer.decode(a, skip_special_tokens=True))

BEST = dict(k=53, n_mut=7, n_top=512, n_cand=256, pool_kind="full",
            position="suffix", n_new=48, refresh_every=2, init="repeat")
print("phase 6 trial 12 params:", BEST)
print("reference (FULL metric, out.cent): unsteered ctrl 0.0499-0.0546 | "
      "real bridge q 0.0864-0.1025 | phase 6 GCG winner 0.0620")
print("this run is scored on the TRUNCATED metric — read it against the truncated "
      "control band printed above.\n")

def _log(s, tm, best, el):
    print(f"  step {s:>3}  true={tm:.4f}  best={best:.4f}  {el:>5.0f}s")

t0 = time.time()
final = gcg2(budget_s=300, seed=1, chunk=8, log=_log, **BEST)
print(f"\ndone in {time.time()-t0:.0f}s | steps={final['steps']}")
print(f"true (truncated {METRIC_SPACE}) = {final['true']:.4f}")
print("trigger:", repr(tokenizer.decode(final["trigger"])[:200]))
print("answer :", repr(final["answer"][:300]))


phase 6 trial 12 params: {'k': 53, 'n_mut': 7, 'n_top': 512, 'n_cand': 256, 'pool_kind': 'full', 'position': 'suffix', 'n_new': 48, 'refresh_every': 2, 'init': 'repeat'}
reference (FULL metric, out.cent): unsteered ctrl 0.0499-0.0546 | real bridge q 0.0864-0.1025 | phase 6 GCG winner 0.0620
this run is scored on the TRUNCATED metric — read it against the truncated control band printed above.

  step   2  true=0.0460  best=0.0466      9s
  step   4  true=0.0481  best=0.0481     18s
  step   6  true=0.0501  best=0.0501     27s
  step   8  true=0.0513  best=0.0513     36s
  step  10  true=0.0572  best=0.0572     45s
  step  12  true=0.0482  best=0.0572     54s
  step  14  true=0.0558  best=0.0572     64s
  step  16  true=0.0531  best=0.0572     73s
  step  18  true=0.0528  best=0.0572     82s
  step  20  true=0.0527  best=0.0572     91s
  step  22  true=0.0583  best=0.0583    100s
  step  24  true=0.0597  best=0.0597    109s
  step  26  true=0.0573  best=0.0597    118s
  step  28  true=0.

In [9]:
# === §7 — four-space control, both metrics, and the phase 6 trigger for comparison ===
#
# RECIPE stage 6: score the winner in every space, not just the optimised one. Two extra
# columns here — the same trigger under the FULL metric, and phase 6's own winning trigger
# rescored under the truncated one, so the two runs are comparable in both directions.
import torch, json

PHASE6_TRIGGER = [1597, 61300, 48673, 67136, 56568, 11135, 4326, 24626, 117330, 109656,
                  70591, 31047, 33008, 20931, 146445, 10813, 38573, 70040, 128633, 133675,
                  10806, 56472, 99679, 51133, 36075, 2447, 23747, 49153, 112618, 33086,
                  2675, 44387, 31031, 23757, 2595, 32794, 2934, 6835, 79456, 111662, 458,
                  75610, 28469, 1877, 2585, 20706, 86, 9167, 71231, 90019, 119112, 104655,
                  99470]

@torch.no_grad()
def score_trigger_all(trig, q="what shall i do today", position="suffix", n_new=45):
    PRE, SUF = make_scaffold(q, position)
    ans = rollout(trig, PRE, SUF, n_new)
    seq = torch.cat([torch.tensor(PRE, device=dev), trig.to(dev),
                     torch.tensor(SUF, device=dev), ans])
    lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    P  = lg.softmax(-1); del lg
    vals, idx = P.topk(K_KEEP, dim=-1)
    den = vals.sum(-1)
    out = {}
    for k in KEYS:
        out[f"{k}|full"]  = (P @ COS[k]).mean().item()
        out[f"{k}|trunc"] = ((vals * COS[k][idx]).sum(-1) / den).mean().item()
    return out, tokenizer.decode(ans, skip_special_tokens=True), \
           len(set(ans.tolist())) / len(ans), den.mean().item()

new_scores, new_ans, new_dist, new_mass = score_trigger_all(final["trigger"])
p6 = torch.tensor(PHASE6_TRIGGER)
p6_scores, p6_ans, p6_dist, p6_mass = score_trigger_all(p6)

print(f"{'space':<9} {'metric':<6} {'§7 GCG':>9} {'phase6 GCG':>11} "
      f"{'ctrl band':>20} {'bridge q':>20}")
print("-" * 82)
for k in KEYS:
    for m in ("full", "trunc"):
        cb = [summary[q][k][m] for q in QUERIES[:4]]
        bb = [summary[q][k][m] for q in QUERIES[4:]]
        star = " <-- optimised" if (k == METRIC_SPACE and m == "trunc") else ""
        print(f"{k:<9} {m:<6} {new_scores[f'{k}|{m}']:>9.4f} {p6_scores[f'{k}|{m}']:>11.4f} "
              f"[{min(cb):+.4f},{max(cb):+.4f}] [{min(bb):+.4f},{max(bb):+.4f}]{star}")

print(f"\n§7 trigger : distinct={new_dist:.2f} mass_kept={new_mass:.4f}")
print(f"  {new_ans[:300]!r}")
print(f"\nphase 6 trigger rescored: distinct={p6_dist:.2f} mass_kept={p6_mass:.4f}")
print(f"  {p6_ans[:300]!r}")

res = dict(
    meta=dict(model=MODEL_ID, metric_space=METRIC_SPACE, truncation="top-quartile",
              K_KEEP=K_KEEP, vocab=V, thinking=False, params=BEST, budget_s=300, seed=1),
    separation=summary,
    gcg=dict(true=final["true"], steps=final["steps"],
             trigger_ids=final["trigger"].tolist(),
             trigger=tokenizer.decode(final["trigger"]),
             answer=new_ans, distinct=new_dist, all_spaces=new_scores),
    phase6_trigger=dict(trigger_ids=PHASE6_TRIGGER, answer=p6_ans,
                        distinct=p6_dist, all_spaces=p6_scores),
)
with open("phase6_top25_results.json", "w") as f:
    json.dump(res, f, indent=1)
print("\nsaved phase6_top25_results.json")


space     metric    §7 GCG  phase6 GCG            ctrl band             bridge q
----------------------------------------------------------------------------------
in.raw    full      0.0388      0.0386 [+0.0359,+0.0405] [+0.0549,+0.0782]
in.raw    trunc     0.0388      0.0386 [+0.0359,+0.0405] [+0.0549,+0.0782]
in.cent   full      0.0215      0.0216 [+0.0189,+0.0232] [+0.0377,+0.0615]
in.cent   trunc     0.0215      0.0216 [+0.0189,+0.0232] [+0.0377,+0.0615]
out.raw   full     -0.0225     -0.0212 [-0.0216,-0.0158] [+0.0137,+0.0379]
out.raw   trunc    -0.0225     -0.0212 [-0.0216,-0.0158] [+0.0137,+0.0379]
out.cent  full      0.0595      0.0620 [+0.0499,+0.0546] [+0.0864,+0.1025]
out.cent  trunc     0.0595      0.0620 [+0.0499,+0.0546] [+0.0864,+0.1025] <-- optimised

§7 trigger : distinct=0.73 mass_kept=1.0000
  "It looks like your message is a mix of random words, names, and phrases that don't form a coherent question or statement. It might be a result of a typo, a mix-up of differen

In [12]:
# === §8 — the nucleus cut: keep only tokens the model might actually emit ===
#
# The quartile cut dropped 113,952 tokens and moved the score by 4e-9 — they carried no
# mass at all, so it never tested the real question. This is the sharp version: keep the
# smallest set of tokens whose probabilities sum to p_cut, renormalised. Everything
# outside is plausible-but-unrealised vocabulary — where a Goodharting optimiser would
# hide if it hides anywhere.
#
# Implemented over a top-NUC_K window rather than a full sort (sorting [B*T, 151936] per
# candidate batch is unaffordable). The window is asserted never to saturate.
import torch

NUC_K = 4096

def nucleus_parts(P, p_cut, K=NUC_K):
    """P [..., V] -> (weights w [..., K], gathered ids idx, nucleus size). Keeps >=1 token."""
    vals, idx = P.topk(K, dim=-1)                 # descending
    keep = (vals.cumsum(-1) - vals) < p_cut       # token enters iff mass strictly before it < p_cut
    return vals * keep, idx, keep.sum(-1)

def nucleus_score(P, cos, p_cut, K=NUC_K):
    w, idx, n = nucleus_parts(P, p_cut, K)
    return (w * cos[idx]).sum(-1) / w.sum(-1), n

CUTS = [0.99, 0.9, 0.5]
nuc_summary = {}
print(f"{'query':<46} {'cut':>5} {'|nuc| mean':>11} {'max':>5} "
      + " ".join(f"{k:>9}" for k in KEYS))
print("-" * 118)
for q in QUERIES:
    P, ans = whole_answer_probs(q)
    row = {}
    for c in CUTS:
        w, idx, n = nucleus_parts(P, c)
        assert int(n.max()) < NUC_K, f"nucleus saturated the {NUC_K} window at p={c}"
        den = w.sum(-1)
        sc = {k: ((w * COS[k][idx]).sum(-1) / den).mean().item() for k in KEYS}
        sc["nuc_mean"] = n.float().mean().item()
        sc["nuc_max"]  = int(n.max())
        row[c] = sc
        print(f"{q if c == CUTS[0] else '':<46} {c:>5} {sc['nuc_mean']:>11.1f} "
              f"{sc['nuc_max']:>5} " + " ".join(f"{sc[k]:>9.4f}" for k in KEYS))
        del w, idx, den
    nuc_summary[q] = row
    del P
    torch.cuda.empty_cache()
    print("-" * 118)

print("\nseparation check under each cut (control band vs bridge queries):")
for c in CUTS:
    for k in KEYS:
        cb = [nuc_summary[q][c][k] for q in QUERIES[:4]]
        bb = [nuc_summary[q][c][k] for q in QUERIES[4:]]
        ok = "OK  " if min(bb) > max(cb) else "FAIL"
        print(f"  p={c:<5} {k:<9} controls [{min(cb):+.4f}, {max(cb):+.4f}]  "
              f"bridge [{min(bb):+.4f}, {max(bb):+.4f}]  {ok}")
    print()


query                                            cut  |nuc| mean   max    in.raw   in.cent   out.raw  out.cent
----------------------------------------------------------------------------------------------------------------------
what shall i do today                           0.99         4.8   101    0.0369    0.0198   -0.0216    0.0546
                                                 0.9         2.0    18    0.0369    0.0198   -0.0217    0.0547
                                                 0.5         1.1     4    0.0374    0.0203   -0.0222    0.0550
----------------------------------------------------------------------------------------------------------------------
recommend me a book                             0.99         2.5    14    0.0359    0.0188   -0.0180    0.0516
                                                 0.9         1.6     5    0.0358    0.0188   -0.0180    0.0515
                                                 0.5         1.1     2    0.0357    0.0187   -0.

In [13]:
# === §8 — GCG against the p=0.99 nucleus metric ===
#
# score_batch / grad_onehot rebound to the nucleus form; rollout, true_metric and gcg2 call
# them by global name, so everything downstream picks this up unchanged. The gradient can
# now only flow through the ~3-5 tokens per position that hold 99% of the mass — three
# orders of magnitude fewer paths than the quartile cut left open.
import torch, torch.nn.functional as F, time

NUC_P = 0.99

@torch.no_grad()
def score_batch(trigs, PRE, SUF, ans, chunk=8):
    """mean_t E[cos | v in the p=NUC_P nucleus], teacher-forced on `ans`. [B, k] -> [B]"""
    B, na = trigs.shape[0], len(ans)
    pre = torch.tensor(PRE, device=dev); suf = torch.tensor(SUF, device=dev)
    out = []
    for i in range(0, B, chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1),
                         ans.expand(b, -1)], dim=1)
        lg = model(seq, **{_LTK: na + 1}).logits[:, :-1].float()
        p  = lg.softmax(-1); del lg
        w, idx, _ = nucleus_parts(p, NUC_P); del p
        out.append(((w * TARGET_COS[idx]).sum(-1) / w.sum(-1)).mean(-1))
        del w, idx
    return torch.cat(out)

def grad_onehot(trig, PRE, SUF, ans):
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), num_classes=V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE, device=dev)], oh @ E,
                     E[torch.tensor(SUF, device=dev)], E[ans]]).unsqueeze(0)
    lg = model(inputs_embeds=inp, **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    p  = lg.softmax(-1)
    w, idx, _ = nucleus_parts(p, NUC_P)
    ((w * TARGET_COS[idx]).sum(-1) / w.sum(-1)).mean().backward()
    g = oh.grad.detach().clone()
    del oh, inp, lg, p, w, idx
    torch.cuda.empty_cache()
    return g

# sanity: the rebound objective must reproduce the nucleus numbers measured above
_chk = nuc_summary["tell me about bridges"][NUC_P][METRIC_SPACE]
print(f"objective rebound to nucleus p={NUC_P} | reference 'tell me about bridges' "
      f"{METRIC_SPACE} = {_chk:.4f}")
print("control band (p=0.99, out.cent): "
      f"[{min(nuc_summary[q][NUC_P]['out.cent'] for q in QUERIES[:4]):.4f}, "
      f"{max(nuc_summary[q][NUC_P]['out.cent'] for q in QUERIES[:4]):.4f}]  "
      f"| bridge q [{min(nuc_summary[q][NUC_P]['out.cent'] for q in QUERIES[4:]):.4f}, "
      f"{max(nuc_summary[q][NUC_P]['out.cent'] for q in QUERIES[4:]):.4f}]")
print("prior runs, same params: full-vocab 0.0620 | top-quartile 0.0595\n")

t0 = time.time()
final_nuc = gcg2(budget_s=300, seed=1, chunk=8, log=_log, **BEST)
print(f"\ndone in {time.time()-t0:.0f}s | steps={final_nuc['steps']}")
print(f"true (nucleus p={NUC_P}, {METRIC_SPACE}) = {final_nuc['true']:.4f}")
print("trigger:", repr(tokenizer.decode(final_nuc["trigger"])[:200]))
print("answer :", repr(final_nuc["answer"][:300]))


objective rebound to nucleus p=0.99 | reference 'tell me about bridges' out.cent = 0.0864
control band (p=0.99, out.cent): [0.0499, 0.0546]  | bridge q [0.0864, 0.1026]
prior runs, same params: full-vocab 0.0620 | top-quartile 0.0595

  step   2  true=0.0557  best=0.0557      9s
  step   4  true=0.0440  best=0.0557     18s
  step   6  true=0.0554  best=0.0557     27s
  step   8  true=0.0528  best=0.0557     36s
  step  10  true=0.0552  best=0.0557     45s
  step  12  true=0.0562  best=0.0562     54s
  step  14  true=0.0525  best=0.0562     63s
  step  16  true=0.0533  best=0.0562     72s
  step  18  true=0.0525  best=0.0562     81s
  step  20  true=0.0537  best=0.0562     91s
  step  22  true=0.0555  best=0.0562    100s
  step  24  true=0.0598  best=0.0598    109s
  step  26  true=0.0594  best=0.0598    118s
  step  28  true=0.0555  best=0.0598    127s
  step  30  true=0.0556  best=0.0598    136s
  step  32  true=0.0543  best=0.0598    145s
  step  34  true=0.0553  best=0.0598    154s


In [14]:
# === §8 — four-space control, and the three runs side by side ===
import torch, json

@torch.no_grad()
def score_trigger_modes(trig, q="what shall i do today", position="suffix", n_new=45):
    PRE, SUF = make_scaffold(q, position)
    ans = rollout(trig, PRE, SUF, n_new)
    seq = torch.cat([torch.tensor(PRE, device=dev), trig.to(dev),
                     torch.tensor(SUF, device=dev), ans])
    lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    P  = lg.softmax(-1); del lg
    out = {f"{k}|full": (P @ COS[k]).mean().item() for k in KEYS}
    for c in (0.99, 0.9, 0.5):
        w, idx, n = nucleus_parts(P, c); den = w.sum(-1)
        for k in KEYS:
            out[f"{k}|nuc{c}"] = ((w * COS[k][idx]).sum(-1) / den).mean().item()
        out[f"nuc{c}_size"] = n.float().mean().item()
        del w, idx, den
    del P; torch.cuda.empty_cache()
    return out, tokenizer.decode(ans, skip_special_tokens=True), len(set(ans.tolist())) / len(ans)

RUNS = {
    "phase6 (full vocab)":  torch.tensor(PHASE6_TRIGGER),
    "§7 (top quartile)":    final["trigger"],
    "§8 (nucleus p=0.99)":  final_nuc["trigger"],
}
scored = {}
for name, trig in RUNS.items():
    scored[name] = score_trigger_modes(trig)

print("out.cent under every readout (the optimised space):")
print(f"{'run':<22} {'full':>9} {'nuc0.99':>9} {'nuc0.9':>9} {'nuc0.5':>9} {'distinct':>9}")
print("-" * 72)
for name in RUNS:
    s, ans, d = scored[name]
    print(f"{name:<22} {s['out.cent|full']:>9.4f} {s['out.cent|nuc0.99']:>9.4f} "
          f"{s['out.cent|nuc0.9']:>9.4f} {s['out.cent|nuc0.5']:>9.4f} {d:>9.2f}")
cb = [summary[q]["out.cent"]["full"] for q in QUERIES[:4]]
bb = [summary[q]["out.cent"]["full"] for q in QUERIES[4:]]
print(f"{'[control band]':<22} [{min(cb):.4f}, {max(cb):.4f}]")
print(f"{'[real bridge query]':<22} [{min(bb):.4f}, {max(bb):.4f}]")

print("\nfour-space control, full readout:")
print(f"{'run':<22} " + " ".join(f"{k:>9}" for k in KEYS))
print("-" * 62)
for name in RUNS:
    s, _, _ = scored[name]
    print(f"{name:<22} " + " ".join(f"{s[f'{k}|full']:>9.4f}" for k in KEYS))
for lbl, qs in (("[control band]", QUERIES[:4]), ("[bridge query]", QUERIES[4:])):
    lo = [f"{min(summary[q][k]['full'] for q in qs):>+9.4f}" for k in KEYS]
    hi = [f"{max(summary[q][k]['full'] for q in qs):>+9.4f}" for k in KEYS]
    print(f"{lbl+' lo':<22} " + " ".join(lo))
    print(f"{lbl+' hi':<22} " + " ".join(hi))

print("\nanswers:")
for name in RUNS:
    _, ans, _ = scored[name]
    print(f"\n  {name}\n    {ans[:220]!r}")

res8 = dict(
    meta=dict(model=MODEL_ID, metric_space=METRIC_SPACE, params=BEST, seed=1,
              budget_s=300, nucleus_p=NUC_P, nucleus_window=NUC_K),
    nucleus_separation=nuc_summary,
    runs={name: dict(trigger_ids=RUNS[name].tolist(), all_modes=scored[name][0],
                     answer=scored[name][1], distinct=scored[name][2])
          for name in RUNS},
    gcg_nucleus=dict(true=final_nuc["true"], steps=final_nuc["steps"]),
)
with open("phase6_nucleus_results.json", "w") as f:
    json.dump(res8, f, indent=1)
print("\nsaved phase6_nucleus_results.json")


out.cent under every readout (the optimised space):
run                         full   nuc0.99    nuc0.9    nuc0.5  distinct
------------------------------------------------------------------------
phase6 (full vocab)       0.0620    0.0621    0.0623    0.0624      0.78
§7 (top quartile)         0.0595    0.0595    0.0601    0.0600      0.73
§8 (nucleus p=0.99)       0.0572    0.0573    0.0575    0.0579      0.89
[control band]         [0.0499, 0.0546]
[real bridge query]    [0.0864, 0.1025]

four-space control, full readout:
run                       in.raw   in.cent   out.raw  out.cent
--------------------------------------------------------------
phase6 (full vocab)       0.0386    0.0216   -0.0212    0.0620
§7 (top quartile)         0.0388    0.0215   -0.0225    0.0595
§8 (nucleus p=0.99)       0.0378    0.0209   -0.0182    0.0572
[control band] lo        +0.0359   +0.0189   -0.0216   +0.0499
[control band] hi        +0.0405   +0.0232   -0.0158   +0.0546
[bridge query] lo        +0

In [15]:
# === §9 — Optuna over phase 6's space PLUS the fraction of probability mass discarded ===
#
# `discard` is the share of probability mass excluded from the objective: the search sees
# only the (1 - discard) nucleus at each position. discard=0 is phase 6's full-vocabulary
# metric; discard=0.9 leaves the search looking at ~1 token per position.
#
# ⚠ Scores at different cutoffs are not the same number, so maximising each trial's OWN
# objective would just select the cutoff with the largest scale. Every trial is therefore
# searched at its own cutoff and then scored on a COMMON yardstick — the full-vocabulary
# metric on a fresh greedy rollout — which is the number Optuna maximises.
!pip install -q optuna
import optuna, torch, torch.nn.functional as F, time
optuna.logging.set_verbosity(optuna.logging.WARNING)

DISCARDS  = [0.0, 0.001, 0.01, 0.05, 0.2, 0.5, 0.9]
NUC_W     = 8192          # window; wide enough for discard=0.001
CUR_PCUT  = 1.0
_nuc_warn = 0

def obj_from_probs(p):
    """p [..., V] -> mean-over-positions score under the CURRENT trial's cutoff."""
    global _nuc_warn
    if CUR_PCUT >= 1.0:
        return (p @ TARGET_COS).mean(-1)
    vals, idx = p.topk(NUC_W, dim=-1)
    keep = (vals.cumsum(-1) - vals) < CUR_PCUT
    if bool(keep[..., -1].any()):
        _nuc_warn += 1                      # nucleus hit the window; score is truncated
    w = vals * keep
    return ((w * TARGET_COS[idx]).sum(-1) / w.sum(-1)).mean(-1)

@torch.no_grad()
def score_batch(trigs, PRE, SUF, ans, chunk=8):
    B, na = trigs.shape[0], len(ans)
    pre = torch.tensor(PRE, device=dev); suf = torch.tensor(SUF, device=dev)
    out = []
    for i in range(0, B, chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1), ans.expand(b, -1)], dim=1)
        lg = model(seq, **{_LTK: na + 1}).logits[:, :-1].float()
        p  = lg.softmax(-1); del lg
        out.append(obj_from_probs(p)); del p
    return torch.cat(out)

def grad_onehot(trig, PRE, SUF, ans):
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), num_classes=V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE, device=dev)], oh @ E,
                     E[torch.tensor(SUF, device=dev)], E[ans]]).unsqueeze(0)
    lg = model(inputs_embeds=inp, **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    obj_from_probs(lg.softmax(-1)).backward()
    g = oh.grad.detach().clone()
    del oh, inp, lg
    torch.cuda.empty_cache()
    return g

@torch.no_grad()
def yardstick(trig, position, q="what shall i do today", n_new=45):
    """common scale for every trial: the FULL-vocabulary metric on a fresh rollout"""
    PRE, SUF = make_scaffold(q, position)
    ans = rollout(trig, PRE, SUF, n_new)
    seq = torch.cat([torch.tensor(PRE, device=dev), trig.to(dev),
                     torch.tensor(SUF, device=dev), ans])
    lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    P  = lg.softmax(-1); del lg
    sc = {k: (P @ COS[k]).mean().item() for k in KEYS}
    del P; torch.cuda.empty_cache()
    return sc, tokenizer.decode(ans, skip_special_tokens=True), \
           len(set(ans.tolist())) / len(ans)

TRIAL_S, N_TRIALS = 150, 16

def objective8(t):
    global CUR_PCUT
    discard  = t.suggest_categorical("discard", DISCARDS)
    CUR_PCUT = 1.0 - discard
    p = dict(
        k             = t.suggest_int("k", 16, 254, log=True),
        n_mut         = t.suggest_int("n_mut", 1, 8),
        n_top         = t.suggest_categorical("n_top", [64, 128, 256, 512, 1024]),
        n_cand        = t.suggest_categorical("n_cand", [64, 128, 256]),
        pool_kind     = t.suggest_categorical("pool_kind", ["weak4096", "weak16384", "full"]),
        position      = t.suggest_categorical("position", ["prefix", "suffix"]),
        n_new         = t.suggest_categorical("n_new", [24, 32, 48]),
        refresh_every = t.suggest_int("refresh_every", 1, 6),
        init          = t.suggest_categorical("init", ["random", "repeat"]),
    )
    try:
        r = gcg2(budget_s=TRIAL_S, seed=1, chunk=8, **p)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); raise optuna.TrialPruned()
    sc, ans, dist = yardstick(r["trigger"], p["position"])
    t.set_user_attr("own_objective", r["true"])       # score at THIS trial's cutoff
    t.set_user_attr("all_spaces", sc)
    t.set_user_attr("trigger", tokenizer.decode(r["trigger"]))
    t.set_user_attr("trigger_ids", r["trigger"].tolist())
    t.set_user_attr("answer", ans[:300])
    t.set_user_attr("distinct", dist)
    t.set_user_attr("steps", r["steps"])
    print(f"  trial {t.number:>2}  yardstick={sc['out.cent']:.4f}  own={r['true']:.4f}  "
          f"discard={discard:<6} steps={r['steps']:>3}  k={p['k']:>3} mut={p['n_mut']} "
          f"cand={p['n_cand']} pool={p['pool_kind']:<10} {p['position']}", flush=True)
    return sc["out.cent"]

print(f"Optuna: {N_TRIALS} trials x {TRIAL_S}s ~= {N_TRIALS*(TRIAL_S+8)/60:.0f} min")
print("yardstick = FULL-vocab out.cent | control band 0.0499-0.0546 | "
      "real bridge q 0.0864-0.1025")
print("fixed-param references at this yardstick: full 0.0620 | quartile 0.0595 | "
      "nucleus0.99 0.0572\n", flush=True)

t0 = time.time()
study8 = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(seed=1))
study8.optimize(objective8, n_trials=N_TRIALS)
print(f"\ndone in {(time.time()-t0)/60:.1f} min | window warnings: {_nuc_warn}")
print(f"best yardstick: {study8.best_value:.4f}")
print("best params:", study8.best_params)
print("trigger:", repr(study8.best_trial.user_attrs["trigger"][:200]))
print("answer :", repr(study8.best_trial.user_attrs["answer"][:200]))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.4 MB/s eta 0:00:00
Optuna: 16 trials x 150s ~= 42 min
yardstick = FULL-vocab out.cent | control band 0.0499-0.0546 | real bridge q 0.0864-0.1025
fixed-param references at this yardstick: full 0.0620 | quartile 0.0595 | nucleus0.99 0.0572

  trial  0  yardstick=0.0581  own=0.0581  discard=0.001  steps= 71  k= 41 mut=4 cand=128 pool=weak4096   suffix
  trial  1  yardstick=0.0606  own=0.0614  discard=0.2    steps= 69  k= 37 mut=6 cand=128 pool=weak16384  prefix
  trial  2  yardstick=0.0559  own=0.0562  discard=0.05   steps= 92  k= 49 mut=1 cand=64 pool=weak4096   prefix
  trial  3  yardstick=0.0545  own=0.0547  discard=0.05   steps= 43  k=221 mut=4 cand=128 pool=weak16384  prefix
  trial  4  yardstick=0.0632  own=0.0632  discard=0.0    steps= 70  k= 22 mut=1 cand=128 pool=full       prefix


[W 2026-08-03 14:10:25,284] Trial 5 failed with parameters: {'discard': 0.0, 'k': 114, 'n_mut': 5, 'n_top': 256, 'n_cand': 128, 'pool_kind': 'weak16384', 'position': 'suffix', 'n_new': 32, 'refresh_every': 1, 'init': 'repeat'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_462/2143037046.py", line 89, in objective8
    r = gcg2(budget_s=TRIAL_S, seed=1, chunk=8, **p)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_462/127900347.py", line 34, in gcg2
    sc = score_batch(cands, PRE, SUF, ans, chunk=chunk)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
 

KeyboardInterrupt: 

In [16]:
# === §10 — controlled sweep: discard is the ONLY variable, with seed replication ===
#
# Everything else fixed at phase 6 trial 12's parameters. Three seeds per level, so the
# endpoint's seed variance is measured rather than assumed. Seed enters both the trigger
# init and the mutation draws, so it is the whole stochastic content of a run.
#
# Yardstick: full-vocabulary out.cent on a fresh greedy rollout. Reported at n_new=160,
# which is the length §2's control band (0.0499-0.0546) was measured at, so the comparison
# is like-for-like; n_new=45 is carried alongside because the earlier runs used it.
import torch, json, time, statistics as st

DISCARDS = [0.0, 0.001, 0.01, 0.05, 0.2, 0.5, 0.9]
SEEDS    = [1, 2, 3]
SWEEP_S  = 150

@torch.no_grad()
def yardstick_n(trig, position, q="what shall i do today", n_new=160):
    PRE, SUF = make_scaffold(q, position)
    ans = rollout(trig, PRE, SUF, n_new)
    seq = torch.cat([torch.tensor(PRE, device=dev), trig.to(dev),
                     torch.tensor(SUF, device=dev), ans])
    lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    P  = lg.softmax(-1); del lg
    sc = {k: (P @ COS[k]).mean().item() for k in KEYS}
    del P; torch.cuda.empty_cache()
    return sc, tokenizer.decode(ans, skip_special_tokens=True), \
           len(set(ans.tolist())) / len(ans)

rows, t_all = [], time.time()
print(f"{len(DISCARDS)}x{len(SEEDS)} = {len(DISCARDS)*len(SEEDS)} runs x {SWEEP_S}s "
      f"~= {len(DISCARDS)*len(SEEDS)*(SWEEP_S+12)/60:.0f} min")
print(f"fixed params: {BEST}")
print("control band @160 = 0.0499-0.0546 | real bridge q = 0.0864-0.1025\n")
print(f"{'discard':>8} {'seed':>5} {'y@160':>8} {'y@45':>8} {'own':>8} {'steps':>6} {'dist':>6}",
      flush=True)

for d in DISCARDS:
    global CUR_PCUT
    CUR_PCUT = 1.0 - d
    for sd in SEEDS:
        r = gcg2(budget_s=SWEEP_S, seed=sd, chunk=8, **BEST)
        s160, ans160, dist160 = yardstick_n(r["trigger"], BEST["position"], n_new=160)
        s45,  ans45,  _       = yardstick_n(r["trigger"], BEST["position"], n_new=45)
        rows.append(dict(discard=d, seed=sd, y160=s160["out.cent"], y45=s45["out.cent"],
                         own=r["true"], steps=r["steps"], distinct=dist160,
                         spaces160=s160, trigger_ids=r["trigger"].tolist(),
                         answer=ans160[:300]))
        print(f"{d:>8} {sd:>5} {s160['out.cent']:>8.4f} {s45['out.cent']:>8.4f} "
              f"{r['true']:>8.4f} {r['steps']:>6} {dist160:>6.2f}", flush=True)
        with open("phase6_discard_sweep.json", "w") as f:
            json.dump(dict(meta=dict(params=BEST, budget_s=SWEEP_S, seeds=SEEDS,
                                     discards=DISCARDS), rows=rows), f, indent=1)

print(f"\ntotal {(time.time()-t_all)/60:.1f} min\n")
print(f"{'discard':>8} {'mean y@160':>11} {'sd':>8} {'min':>8} {'max':>8}")
print("-" * 47)
for d in DISCARDS:
    v = [r["y160"] for r in rows if r["discard"] == d]
    print(f"{d:>8} {st.mean(v):>11.4f} {st.stdev(v):>8.4f} {min(v):>8.4f} {max(v):>8.4f}")

allv = [r["y160"] for r in rows]
within = st.mean([st.stdev([r["y160"] for r in rows if r["discard"] == d]) for d in DISCARDS])
between = st.stdev([st.mean([r["y160"] for r in rows if r["discard"] == d]) for d in DISCARDS])
print(f"\nmean within-level sd (seed noise) : {within:.4f}")
print(f"sd of level means (discard effect): {between:.4f}")
print(f"=> discard effect is {'LARGER' if between > within else 'SMALLER'} than seed noise")
print(f"overall: mean {st.mean(allv):.4f} range {min(allv):.4f}-{max(allv):.4f}")
print("\nsaved phase6_discard_sweep.json")


7x3 = 21 runs x 150s ~= 57 min
fixed params: {'k': 53, 'n_mut': 7, 'n_top': 512, 'n_cand': 256, 'pool_kind': 'full', 'position': 'suffix', 'n_new': 48, 'refresh_every': 2, 'init': 'repeat'}
control band @160 = 0.0499-0.0546 | real bridge q = 0.0864-0.1025

 discard  seed    y@160     y@45      own  steps   dist
     0.0     1   0.0595   0.0596   0.0596     34   0.61
     0.0     2   0.0582   0.0603   0.0603     34   0.66
     0.0     3   0.0556   0.0604   0.0604     34   0.63
   0.001     1   0.0579   0.0581   0.0581     33   0.60
   0.001     2   0.0562   0.0602   0.0603     33   0.63
   0.001     3   0.0549   0.0579   0.0579     33   0.66
    0.01     1   0.0519   0.0590   0.0590     33   0.65
    0.01     2   0.0530   0.0605   0.0606     33   0.70
    0.01     3   0.0571   0.0592   0.0592     34   0.54
    0.05     1   0.0451   0.0595   0.0597     33   0.52
    0.05     2   0.0448   0.0591   0.0595     33   0.77
    0.05     3   0.0481   0.0557   0.0559     33   0.71
     0.2     1 

In [17]:
# === §11 — the positive control: steering, scored under every readout ===
#
# RECIPE stage 6's last paragraph: score the intervention that ACTUALLY produces the
# behaviour on the same objective. The truncated and nucleus metrics have so far only been
# validated on natural text (real bridge questions vs controls). This exercises the other
# half of the range — the degenerate regime phase 6 §3 showed the full metric rewards.
#
# Protocol is phase 6 §3's verbatim: CAA over 8 negatives with the shared "The word is"
# context prefix (bare single-token pairs land on the attention sink), L16, alpha scaled to
# the non-sink residual norm, injected at every position but 0, held on during generation,
# T=0.8 / 45 tokens / 4 samples, seeds 0-3. in.cent|full should reproduce phase 6's §3 table.
import torch, torch.nn.functional as F, json, time
from contextlib import contextmanager

LAYERS, L_STEER, CTX, POS = model.model.layers, 16, "The word is", " bridge"

def _ids(t): return tokenizer(t, add_special_tokens=False).input_ids
def _chat(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)

_n = len(_ids(CTX + POS))
NEGS = [n for n in [" cat", " chair", " cloud", " music", " running",
                    " table", " coffee", " window", " paper", " orange"]
        if len(_ids(CTX + n)) == _n][:8]

@torch.no_grad()
def last_tok_vector(pos_txt, neg_txt, layer, ctx=CTX):
    a, b = _ids(ctx + pos_txt), _ids(ctx + neg_txt)
    ha = model(torch.tensor([a], device=dev), output_hidden_states=True).hidden_states[layer][0]
    hb = model(torch.tensor([b], device=dev), output_hidden_states=True).hidden_states[layer][0]
    return (ha[-1] - hb[-1]).float()

SINGLES = torch.stack([last_tok_vector(POS, n, L_STEER) for n in NEGS])
V_CAA   = SINGLES.mean(0)
_N = F.normalize(SINGLES, dim=-1) @ F.normalize(SINGLES, dim=-1).T
_off = _N[~torch.eye(len(NEGS), dtype=bool, device=_N.device)]
print(f"negatives: {NEGS}")
print(f"||v|| single mean {SINGLES.norm(dim=-1).mean():.1f} -> CAA {V_CAA.norm():.1f} | "
      f"pairwise cos {_off.mean():.3f}   (phase 6 measured 69.4 -> 50.0, cos 0.452)")

@torch.no_grad()
def nonsink_norm(text, layer):
    enc = tokenizer(text, return_tensors="pt").to(dev)
    h = model(**enc, output_hidden_states=True).hidden_states[layer][0].float()
    return h[1:].norm(dim=-1).mean().item()

@contextmanager
def steer_at(layer, v, alpha):
    vv = (alpha * v).to(model.dtype)
    def hook(mod, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        h = h.clone()
        if h.shape[1] > 1: h[:, 1:] += vv
        else:              h += vv
        if args: return (h,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h
        return args, kwargs
    hd = LAYERS[layer].register_forward_pre_hook(hook, with_kwargs=True)
    try: yield
    finally: hd.remove()

READOUTS = ["full", "quart", "nuc0.99", "nuc0.9", "nuc0.5"]
SPACES2  = ["in.cent", "out.cent"]

@torch.no_grad()
def all_readouts(Pa):
    out = {}
    for k in SPACES2:
        out[f"{k}|full"] = (Pa @ COS[k]).mean().item()
    vals, idx = Pa.topk(K_KEEP, dim=-1, sorted=False); den = vals.sum(-1)
    for k in SPACES2:
        out[f"{k}|quart"] = ((vals * COS[k][idx]).sum(-1) / den).mean().item()
    del vals, idx, den
    for c in (0.99, 0.9, 0.5):
        w, i2, n = nucleus_parts(Pa, c); d2 = w.sum(-1)
        for k in SPACES2:
            out[f"{k}|nuc{c}"] = ((w * COS[k][i2]).sum(-1) / d2).mean().item()
        out[f"nuc{c}_size"] = n.float().mean().item()
        del w, i2, d2
    return out

@torch.no_grad()
def sample_steered(q, s, max_new=45, n=4, seed=0):
    enc = tokenizer(_chat(q), return_tensors="pt").to(dev)
    n_p = enc.input_ids.shape[1]
    alpha = s * nonsink_norm(_chat(q), L_STEER) / V_CAA.norm().item() if s else 0.0
    per, outs, dist = [], [], []
    for i in range(n):
        torch.manual_seed(seed + i)
        ctx = steer_at(L_STEER, V_CAA, alpha) if s else torch.no_grad()
        with ctx:
            gen = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                                 temperature=0.8, top_p=0.95,
                                 pad_token_id=tokenizer.eos_token_id)[0]
            lg = model(gen.unsqueeze(0)).logits[0, n_p - 1: len(gen) - 1].float()
        ans = gen[n_p:]
        per.append(all_readouts(lg.softmax(-1)))
        outs.append(tokenizer.decode(ans, skip_special_tokens=True))
        dist.append(len(set(ans.tolist())) / max(1, len(ans)))
        del lg
    agg = {kk: sum(p[kk] for p in per) / n for kk in per[0]}
    return outs, agg, sum(dist) / n

CONTROLS  = QUERIES[:4]
STRENGTHS = [0.0, 0.4, 0.6, 0.8, 1.0]
STEER_SWEEP, t0 = {}, time.time()

print(f"\n{'query':<40} {'s':>4} {'dist':>5} " +
      " ".join(f"{r:>8}" for r in READOUTS) + "   (out.cent)")
print("-" * 108)
for q in CONTROLS:
    for s in STRENGTHS:
        outs, agg, dr = sample_steered(q, s)
        STEER_SWEEP[f"{q}|s={s}"] = dict(scores=agg, distinct=dr, samples=outs)
        flag = "  <-- looping" if dr < 0.45 else ""
        print(f"{q[:40]:<40} {s:>4.1f} {dr:>5.2f} " +
              " ".join(f"{agg[f'out.cent|{r}']:>8.4f}" for r in READOUTS) + flag, flush=True)
    print()
    torch.cuda.empty_cache()

print(f"elapsed {(time.time()-t0)/60:.1f} min\n")
print("in.cent, for comparison with phase 6 §3's table (which used in.cent|full):")
print(f"{'query':<40} " + " ".join(f"{'s='+str(s):>8}" for s in STRENGTHS))
for q in CONTROLS:
    print(f"{q[:40]:<40} " +
          " ".join(f"{STEER_SWEEP[f'{q}|s={s}']['scores']['in.cent|full']:>8.4f}"
                   for s in STRENGTHS))

# --- the question this cell exists to answer -----------------------------------
print("\n\nGAMEABILITY: degenerate steered output vs a genuine bridge question")
print(f"{'readout':>10} {'worst loop (s>=0.8)':>21} {'real bridge q':>14} {'ratio':>7}")
print("-" * 56)
loops = [v for kk, v in STEER_SWEEP.items()
         if float(kk.split("=")[1]) >= 0.8 and v["distinct"] < 0.60]
for r in READOUTS:
    if not loops: break
    worst = max(v["scores"][f"out.cent|{r}"] for v in loops)
    real  = (summary["explain how suspension bridges work"]["out.cent"]["full"] if r == "full"
             else nuc_summary["explain how suspension bridges work"].get(
                 float(r[3:]), {}).get("out.cent",
                 summary["explain how suspension bridges work"]["out.cent"]["trunc"]))
    print(f"{r:>10} {worst:>21.4f} {real:>14.4f} {worst/real:>7.2f}x")
print(f"\n(loops found: {len(loops)} cells with distinct < 0.60 at s >= 0.8)")

with open("phase6_steering_readouts.json", "w") as f:
    json.dump(dict(meta=dict(layer=L_STEER, negatives=NEGS, ctx=CTX,
                             caa_norm=V_CAA.norm().item(), pairwise_cos=_off.mean().item(),
                             strengths=STRENGTHS, readouts=READOUTS),
                   sweep=STEER_SWEEP), f, indent=1)
print("saved phase6_steering_readouts.json")


negatives: [' cat', ' chair', ' cloud', ' music', ' running', ' table', ' coffee', ' window']
||v|| single mean 69.4 -> CAA 50.0 | pairwise cos 0.452   (phase 6 measured 69.4 -> 50.0, cos 0.452)

query                                       s  dist     full    quart  nuc0.99   nuc0.9   nuc0.5   (out.cent)
------------------------------------------------------------------------------------------------------------
what shall i do today                     0.0  0.84   0.0514   0.0514   0.0515   0.0515   0.0517
what shall i do today                     0.4  0.81   0.0553   0.0553   0.0554   0.0557   0.0573
what shall i do today                     0.6  0.80   0.0668   0.0668   0.0670   0.0680   0.0679
what shall i do today                     0.8  0.54   0.1356   0.1356   0.1363   0.1412   0.1597
what shall i do today                     1.0  0.40   0.1837   0.1837   0.1854   0.1954   0.2206  <-- looping

recommend me a book                       0.0  0.88   0.0508   0.0508   0.0508   0.051

ValueError: could not convert string to float: 'rt'

In [18]:
# === §11b — gameability table (fixes the readout-name parse in §11) and save ===
import json

REF = {  # 'explain how suspension bridges work', out.cent, under each readout
    "full":    summary["explain how suspension bridges work"]["out.cent"]["full"],
    "quart":   summary["explain how suspension bridges work"]["out.cent"]["trunc"],
    "nuc0.99": nuc_summary["explain how suspension bridges work"][0.99]["out.cent"],
    "nuc0.9":  nuc_summary["explain how suspension bridges work"][0.9]["out.cent"],
    "nuc0.5":  nuc_summary["explain how suspension bridges work"][0.5]["out.cent"],
}
CTRL = {r: max(v["scores"][f"out.cent|{r}"]
               for kk, v in STEER_SWEEP.items() if kk.endswith("s=0.0"))
        for r in READOUTS}

loops = {kk: v for kk, v in STEER_SWEEP.items()
         if float(kk.split("=")[1]) >= 0.8 and v["distinct"] < 0.60}
fluent = {kk: v for kk, v in STEER_SWEEP.items()
          if abs(float(kk.split("=")[1]) - 0.6) < 1e-9}

print("Does truncation make the metric MORE or LESS gameable by repetition?")
print("worst loop = highest-scoring degenerate steered cell (s>=0.8, distinct<0.60)\n")
print(f"{'readout':>9} {'unsteered':>10} {'s=0.6 fluent':>13} {'worst loop':>11} "
      f"{'real bridge q':>14} {'loop/real':>10}")
print("-" * 72)
for r in READOUTS:
    worst = max(v["scores"][f"out.cent|{r}"] for v in loops.values()) if loops else float("nan")
    flu   = max(v["scores"][f"out.cent|{r}"] for v in fluent.values())
    print(f"{r:>9} {CTRL[r]:>10.4f} {flu:>13.4f} {worst:>11.4f} {REF[r]:>14.4f} "
          f"{worst/REF[r]:>10.2f}x")

print(f"\nloops: {len(loops)} cells — {sorted(loops)}")
print(f"\nphase 6 §3 measured this ratio as 0.1749/0.0615 = 2.84x in in.cent|full")
print(f"in.cent|full here: "
      f"{max(v['scores']['in.cent|full'] for v in loops.values())/0.0615:.2f}x")

with open("phase6_steering_readouts.json", "w") as f:
    json.dump(dict(meta=dict(layer=L_STEER, negatives=NEGS, ctx=CTX,
                             caa_norm=V_CAA.norm().item(), pairwise_cos=_off.mean().item(),
                             strengths=STRENGTHS, readouts=READOUTS,
                             reference_bridge_q=REF, unsteered_ceiling=CTRL),
                   sweep=STEER_SWEEP), f, indent=1)
print("\nsaved phase6_steering_readouts.json")


Does truncation make the metric MORE or LESS gameable by repetition?
worst loop = highest-scoring degenerate steered cell (s>=0.8, distinct<0.60)

  readout  unsteered  s=0.6 fluent  worst loop  real bridge q  loop/real
------------------------------------------------------------------------
     full     0.0574        0.0668      0.2132         0.1025       2.08x
    quart     0.0574        0.0668      0.2132         0.1025       2.08x
  nuc0.99     0.0574        0.0670      0.2153         0.1026       2.10x
   nuc0.9     0.0575        0.0680      0.2261         0.1027       2.20x
   nuc0.5     0.0591        0.0679      0.2516         0.1023       2.46x

loops: 6 cells — ['how do I make friends in a new city?|s=1.0', 'recommend me a book|s=1.0', 'what shall i do today|s=0.8', 'what shall i do today|s=1.0', 'what should I get my brother for his birthday?|s=0.8', 'what should I get my brother for his birthday?|s=1.0']

phase 6 §3 measured this ratio as 0.1749/0.0615 = 2.84x in in.cent|f